# Task 2: Pair-Level Sentiment Classification

This notebook implements **Task 2: sentiment classification** for our cannabis project.

## Goal
Predict the sentiment for a **(text, product)** pair, where:

- product ∈ {`flower`, `oil`, `gummies`, `vape`, `topical`}
- sentiment ∈ {`positive`, `negative`, `neutral`}

## Key modeling idea
This is **not** plain text-level sentiment classification.  
It is **product-conditioned sentiment classification**:

- one original text can produce multiple rows
- each row corresponds to one target product
- the model should predict sentiment for **that product in that text**

Example:

Text:
> "Oil helps me sleep, but vape makes me cough."

Pair rows:
- `(text, oil) -> positive`
- `(text, vape) -> negative`

## What this notebook includes
1. Load and validate the pair-level dataset
2. Filter to the final 5 product labels
3. Split train/val/test by `text_id` to avoid leakage
4. Build product-conditioned text inputs
5. Train classical models:
   - TF-IDF + Logistic Regression
   - TF-IDF + LinearSVC
6. Train a transformer model:
   - DistilBERT for 3-class classification
7. Evaluate:
   - accuracy
   - macro F1
   - weighted F1
   - per-class precision/recall/F1
   - confusion matrix
   - multiclass AUROC (if probabilities available)
   - ECE / calibration
8. Save predictions, metrics, and figures

## Important assumption
Your pair-level file should contain at least:
- `text_id`
- `text`
- `product`
- `sentiment`

It can also optionally contain:
- `date_utc`
- `subreddit`
- `source`

This notebook tries to be robust to small schema differences.


In [ ]:
# =========================
# 0. Imports
# =========================
import os
import re
import json
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

FINAL_PRODUCTS = ["flower", "oil", "gummies", "vape", "topical"]
FINAL_SENTIMENTS = ["negative", "neutral", "positive"]  # fixed order for plots / metrics

print("Setup complete.")
print("Products:", FINAL_PRODUCTS)
print("Sentiments:", FINAL_SENTIMENTS)


In [ ]:
# =========================
# 1. Set paths
# =========================
# Edit these if needed.
CANDIDATE_INPUT_PATHS = [
    "./data/final/pair_level_gpt41mini_clean.csv",
    "./data_processing/data/final/pair_level_gpt41mini_clean.csv",
    "../data_processing/data/final/pair_level_gpt41mini_clean.csv",
    "./pair_level_gpt41mini_clean.csv",
]

OUTPUT_DIR = Path("./task2_sentiment_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def first_existing_path(paths):
    for p in paths:
        if Path(p).exists():
            return Path(p)
    return None

PAIR_PATH = first_existing_path(CANDIDATE_INPUT_PATHS)
print("Resolved PAIR_PATH:", PAIR_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


In [ ]:
# =========================
# 2. Load data
# =========================
if PAIR_PATH is None:
    raise FileNotFoundError(
        "Could not find the pair-level CSV. Please edit CANDIDATE_INPUT_PATHS."
    )

df = pd.read_csv(PAIR_PATH)
print("Raw shape:", df.shape)
display(df.head())
print("Columns:", list(df.columns))


In [ ]:
# =========================
# 3. Standardize columns
# =========================
# This cell makes the notebook more robust to small schema differences.

def find_first_matching_column(columns, candidates):
    cols_lower = {c.lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None

text_col = find_first_matching_column(df.columns, ["text", "full_text", "body", "comment", "content"])
text_id_col = find_first_matching_column(df.columns, ["text_id", "id", "reddit_id"])
product_col = find_first_matching_column(df.columns, ["product", "product_label", "product_type"])
sentiment_col = find_first_matching_column(df.columns, ["sentiment", "label", "sentiment_label"])
date_col = find_first_matching_column(df.columns, ["date_utc", "date", "created_utc", "timestamp"])
subreddit_col = find_first_matching_column(df.columns, ["subreddit"])
source_col = find_first_matching_column(df.columns, ["source"])

required = {
    "text_id": text_id_col,
    "text": text_col,
    "product": product_col,
    "sentiment": sentiment_col,
}
missing = [k for k, v in required.items() if v is None]
if missing:
    raise ValueError(f"Missing required columns after schema matching: {missing}")

rename_map = {
    text_id_col: "text_id",
    text_col: "text",
    product_col: "product",
    sentiment_col: "sentiment",
}
if date_col is not None:
    rename_map[date_col] = "date_utc"
if subreddit_col is not None:
    rename_map[subreddit_col] = "subreddit"
if source_col is not None:
    rename_map[source_col] = "source"

df = df.rename(columns=rename_map).copy()

# Keep only needed / useful columns
keep_cols = ["text_id", "text", "product", "sentiment"]
for c in ["date_utc", "subreddit", "source"]:
    if c in df.columns:
        keep_cols.append(c)
df = df[keep_cols].copy()

print("Standardized columns:", list(df.columns))
display(df.head())


In [ ]:
# =========================
# 4. Clean and filter
# =========================
def clean_text_basic(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = x.replace("\n", " ").replace("\r", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x

df["text"] = df["text"].apply(clean_text_basic)
df["product"] = df["product"].astype(str).str.strip().str.lower()
df["sentiment"] = df["sentiment"].astype(str).str.strip().str.lower()

# Keep final product taxonomy and sentiment taxonomy
df = df[df["product"].isin(FINAL_PRODUCTS)].copy()
df = df[df["sentiment"].isin(FINAL_SENTIMENTS)].copy()
df = df[df["text"].str.len() > 0].copy()

# Drop exact duplicate pair rows
df = df.drop_duplicates(subset=["text_id", "text", "product", "sentiment"]).reset_index(drop=True)

# Parse date if present
if "date_utc" in df.columns:
    df["date_utc"] = pd.to_datetime(df["date_utc"], errors="coerce")
    df["year"] = df["date_utc"].dt.year
    df["month"] = df["date_utc"].dt.to_period("M").astype(str)

print("Filtered shape:", df.shape)
print("\nProduct counts:")
print(df["product"].value_counts(dropna=False))
print("\nSentiment counts:")
print(df["sentiment"].value_counts(dropna=False))
display(df.head())


In [ ]:
# =========================
# 5. Quick sanity checks
# =========================
print("Unique text_id:", df["text_id"].nunique())
print("Rows per text_id (top 10):")
display(df.groupby("text_id").size().sort_values(ascending=False).head(10))

print("\nCross-tab: product x sentiment")
display(pd.crosstab(df["product"], df["sentiment"]))

if "month" in df.columns:
    print("\nRows by month (first 12):")
    display(df["month"].value_counts().sort_index().head(12))


## Build product-conditioned input

For sentiment classification, we want the model to know **which product** it should judge inside the text.

We therefore create a new input string like:

```text
[PRODUCT=oil] Oil helps me sleep, but vape makes me cough.
```

This is much better than feeding the raw text alone.


In [ ]:
# =========================
# 6. Product-conditioned input
# =========================
def make_conditioned_text(row):
    return f"[PRODUCT={row['product']}] {row['text']}"

df["model_text"] = df.apply(make_conditioned_text, axis=1)

label2id = {label: i for i, label in enumerate(FINAL_SENTIMENTS)}
id2label = {i: label for label, i in label2id.items()}

df["label_id"] = df["sentiment"].map(label2id)

display(df[["text_id", "product", "sentiment", "model_text"]].head())
print("label2id:", label2id)


## Split by `text_id` to avoid leakage

This is critical.

If the same original text appears in both train and test through different product rows, the evaluation will be overly optimistic.

So we split **unique `text_id`s first**, then bring along all pair rows for those IDs.


In [ ]:
# =========================
# 7. Train/val/test split by text_id
# =========================
unique_ids = pd.Series(df["text_id"].unique()).sample(frac=1.0, random_state=SEED).tolist()

train_ids, temp_ids = train_test_split(unique_ids, test_size=0.30, random_state=SEED)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.50, random_state=SEED)

train_df = df[df["text_id"].isin(train_ids)].reset_index(drop=True)
val_df   = df[df["text_id"].isin(val_ids)].reset_index(drop=True)
test_df  = df[df["text_id"].isin(test_ids)].reset_index(drop=True)

print("Train rows:", train_df.shape, "| unique text_id:", train_df["text_id"].nunique())
print("Val rows:  ", val_df.shape,   "| unique text_id:", val_df["text_id"].nunique())
print("Test rows: ", test_df.shape,  "| unique text_id:", test_df["text_id"].nunique())

# Sanity check: no overlap
assert set(train_df["text_id"]).isdisjoint(set(val_df["text_id"]))
assert set(train_df["text_id"]).isdisjoint(set(test_df["text_id"]))
assert set(val_df["text_id"]).isdisjoint(set(test_df["text_id"]))

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"\n{name} sentiment distribution:")
    print(part["sentiment"].value_counts(normalize=True).round(4))


In [ ]:
# =========================
# 8. Helper functions
# =========================
def multiclass_ece(y_true, y_prob, n_bins=15):
    """
    Expected calibration error for multiclass classification using max confidence.
    y_true: integer labels, shape (n,)
    y_prob: predicted probabilities, shape (n, K)
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    confidences = y_prob.max(axis=1)
    predictions = y_prob.argmax(axis=1)
    accuracies = (predictions == y_true).astype(float)

    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        left, right = bin_edges[i], bin_edges[i + 1]
        if i == n_bins - 1:
            mask = (confidences >= left) & (confidences <= right)
        else:
            mask = (confidences >= left) & (confidences < right)

        if mask.sum() > 0:
            bin_acc = accuracies[mask].mean()
            bin_conf = confidences[mask].mean()
            ece += (mask.mean()) * abs(bin_acc - bin_conf)
    return float(ece)

def safe_multiclass_auroc(y_true, y_prob, average="macro"):
    try:
        return roc_auc_score(y_true, y_prob, multi_class="ovr", average=average)
    except Exception:
        return np.nan

def evaluate_classifier(name, y_true, y_pred, y_prob=None, labels=FINAL_SENTIMENTS, output_dir=OUTPUT_DIR):
    metrics = {}
    metrics["model"] = name
    metrics["accuracy"] = accuracy_score(y_true, y_pred)
    metrics["macro_f1"] = f1_score(y_true, y_pred, average="macro")
    metrics["weighted_f1"] = f1_score(y_true, y_pred, average="weighted")

    pr, rc, f1s, supports = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(len(labels))), zero_division=0
    )
    per_class = pd.DataFrame({
        "class": labels,
        "precision": pr,
        "recall": rc,
        "f1": f1s,
        "support": supports
    })

    if y_prob is not None:
        metrics["macro_auroc_ovr"] = safe_multiclass_auroc(y_true, y_prob, average="macro")
        metrics["weighted_auroc_ovr"] = safe_multiclass_auroc(y_true, y_prob, average="weighted")
        metrics["ece"] = multiclass_ece(y_true, y_prob)
    else:
        metrics["macro_auroc_ovr"] = np.nan
        metrics["weighted_auroc_ovr"] = np.nan
        metrics["ece"] = np.nan

    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(labels))))
    cm_df = pd.DataFrame(cm, index=[f"true_{x}" for x in labels], columns=[f"pred_{x}" for x in labels])

    # Save artifacts
    per_class.to_csv(output_dir / f"{name}_per_class_metrics.csv", index=False)
    cm_df.to_csv(output_dir / f"{name}_confusion_matrix.csv")

    # Plot confusion matrix
    plt.figure(figsize=(6, 5))
    plt.imshow(cm, interpolation="nearest")
    plt.title(f"{name} - Confusion Matrix")
    plt.colorbar()
    tick_marks = np.arange(len(labels))
    plt.xticks(tick_marks, labels, rotation=45)
    plt.yticks(tick_marks, labels)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")
    plt.tight_layout()
    plt.savefig(output_dir / f"{name}_confusion_matrix.png", dpi=200, bbox_inches="tight")
    plt.show()

    print(f"===== {name} =====")
    print(json.dumps(metrics, indent=2))
    print("\nPer-class metrics:")
    display(per_class)
    print("\nClassification report:")
    print(classification_report(
        y_true, y_pred, target_names=labels, zero_division=0
    ))

    return metrics, per_class, cm_df

X_train = train_df["model_text"].tolist()
y_train = train_df["label_id"].tolist()

X_val = val_df["model_text"].tolist()
y_val = val_df["label_id"].tolist()

X_test = test_df["model_text"].tolist()
y_test = test_df["label_id"].tolist()


## Classical Model 1: TF-IDF + Logistic Regression

A strong and interpretable baseline for multiclass text classification.


In [ ]:
# =========================
# 9. TF-IDF + Logistic Regression
# =========================
logreg_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.98,
        sublinear_tf=True,
    )),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=SEED,
        multi_class="auto",
        solver="liblinear",  # robust default
    )),
])

logreg_pipe.fit(X_train, y_train)

val_pred_lr = logreg_pipe.predict(X_val)
test_pred_lr = logreg_pipe.predict(X_test)

# Try to get probabilities
if hasattr(logreg_pipe.named_steps["clf"], "predict_proba"):
    val_prob_lr = logreg_pipe.predict_proba(X_val)
    test_prob_lr = logreg_pipe.predict_proba(X_test)
else:
    val_prob_lr = None
    test_prob_lr = None

logreg_metrics, logreg_pc, logreg_cm = evaluate_classifier(
    "tfidf_logreg", y_test, test_pred_lr, test_prob_lr
)


## Classical Model 2: TF-IDF + LinearSVC

`LinearSVC` is often very competitive for text classification.  
Because raw `LinearSVC` does not produce probabilities, we wrap it with calibration so that we can compute AUROC and ECE too.


In [ ]:
# =========================
# 10. TF-IDF + Calibrated LinearSVC
# =========================
svc_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.98,
        sublinear_tf=True,
    )),
    ("clf", CalibratedClassifierCV(
        estimator=LinearSVC(
            class_weight="balanced",
            random_state=SEED,
        ),
        method="sigmoid",
        cv=3,
    )),
])

svc_pipe.fit(X_train, y_train)

val_pred_svc = svc_pipe.predict(X_val)
test_pred_svc = svc_pipe.predict(X_test)

val_prob_svc = svc_pipe.predict_proba(X_val)
test_prob_svc = svc_pipe.predict_proba(X_test)

svc_metrics, svc_pc, svc_cm = evaluate_classifier(
    "tfidf_calibrated_linearsvc", y_test, test_pred_svc, test_prob_svc
)


In [ ]:
# =========================
# 11. Compare classical models
# =========================
classical_results = pd.DataFrame([logreg_metrics, svc_metrics]).sort_values(
    by=["macro_f1", "accuracy"], ascending=False
).reset_index(drop=True)

display(classical_results)
classical_results.to_csv(OUTPUT_DIR / "classical_model_comparison.csv", index=False)

plt.figure(figsize=(7, 4))
plt.bar(classical_results["model"], classical_results["macro_f1"])
plt.ylabel("Macro F1")
plt.title("Classical Model Comparison")
for i, v in enumerate(classical_results["macro_f1"]):
    plt.text(i, v, f"{v:.3f}", ha="center", va="bottom")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "classical_model_macro_f1.png", dpi=200, bbox_inches="tight")
plt.show()


## Error analysis for the best classical model


In [ ]:
# =========================
# 12. Error analysis helper
# =========================
def build_error_table(df_part, y_true, y_pred, y_prob=None):
    out = df_part.copy()
    out["true_label"] = [id2label[i] for i in y_true]
    out["pred_label"] = [id2label[i] for i in y_pred]
    out["correct"] = out["true_label"] == out["pred_label"]
    if y_prob is not None:
        out["pred_confidence"] = np.max(y_prob, axis=1)
    return out

best_classical_name = (
    "tfidf_logreg" if logreg_metrics["macro_f1"] >= svc_metrics["macro_f1"]
    else "tfidf_calibrated_linearsvc"
)

if best_classical_name == "tfidf_logreg":
    best_classical_errors = build_error_table(test_df, y_test, test_pred_lr, test_prob_lr)
else:
    best_classical_errors = build_error_table(test_df, y_test, test_pred_svc, test_prob_svc)

misclassified = best_classical_errors[~best_classical_errors["correct"]].copy()
misclassified = misclassified.sort_values(
    by="pred_confidence" if "pred_confidence" in misclassified.columns else "text_id",
    ascending=False
)

print("Best classical model:", best_classical_name)
print("Misclassified rows:", misclassified.shape[0])
display(misclassified[["text_id", "product", "sentiment", "true_label", "pred_label", "text"]].head(30))

misclassified.to_csv(OUTPUT_DIR / f"{best_classical_name}_misclassified_examples.csv", index=False)


## Transformer Model: DistilBERT

This section fine-tunes `distilbert-base-uncased` on the pair-level sentiment task.

It uses:
- product-conditioned input text
- 3-class classification head
- evaluation on the validation set at each epoch
- final test evaluation

If `transformers` or `torch` is not installed, this section will raise an import error.


In [ ]:
# =========================
# 13. Build Hugging Face datasets
# =========================
# Skip this section if you only want classical models.

try:
    import torch
    from datasets import Dataset
    from transformers import (
        AutoTokenizer,
        AutoModelForSequenceClassification,
        DataCollatorWithPadding,
        TrainingArguments,
        Trainer,
    )
    HF_AVAILABLE = True
    print("Transformers / torch are available.")
except Exception as e:
    HF_AVAILABLE = False
    print("Transformers section unavailable:", repr(e))


In [ ]:
# =========================
# 14. DistilBERT training
# =========================
if not HF_AVAILABLE:
    print("Skipping transformer training because dependencies are unavailable.")
else:
    model_name = "distilbert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    train_hf = Dataset.from_pandas(
        train_df[["model_text", "label_id"]].rename(columns={"model_text": "text", "label_id": "labels"}),
        preserve_index=False
    )
    val_hf = Dataset.from_pandas(
        val_df[["model_text", "label_id"]].rename(columns={"model_text": "text", "label_id": "labels"}),
        preserve_index=False
    )
    test_hf = Dataset.from_pandas(
        test_df[["model_text", "label_id"]].rename(columns={"model_text": "text", "label_id": "labels"}),
        preserve_index=False
    )

    def tokenize_fn(batch):
        return tokenizer(batch["text"], truncation=True, max_length=256)

    train_hf = train_hf.map(tokenize_fn, batched=True)
    val_hf = val_hf.map(tokenize_fn, batched=True)
    test_hf = test_hf.map(tokenize_fn, batched=True)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(FINAL_SENTIMENTS),
        id2label=id2label,
        label2id=label2id,
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
        preds = probs.argmax(axis=1)
        return {
            "accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro"),
            "weighted_f1": f1_score(labels, preds, average="weighted"),
            "macro_auroc_ovr": safe_multiclass_auroc(labels, probs, average="macro"),
            "ece": multiclass_ece(labels, probs),
        }

    training_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR / "distilbert_ckpt"),
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        logging_dir=str(OUTPUT_DIR / "distilbert_logs"),
        logging_steps=50,
        seed=SEED,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_hf,
        eval_dataset=val_hf,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()


In [ ]:
# =========================
# 15. DistilBERT evaluation
# =========================
if not HF_AVAILABLE:
    print("Skipping transformer evaluation because dependencies are unavailable.")
else:
    pred_output = trainer.predict(test_hf)
    logits = pred_output.predictions
    y_true_hf = pred_output.label_ids
    y_prob_hf = torch.softmax(torch.tensor(logits), dim=1).numpy()
    y_pred_hf = y_prob_hf.argmax(axis=1)

    distilbert_metrics, distilbert_pc, distilbert_cm = evaluate_classifier(
        "distilbert_pair_sentiment",
        y_true_hf,
        y_pred_hf,
        y_prob_hf,
    )

    hf_errors = build_error_table(test_df, y_true_hf, y_pred_hf, y_prob_hf)
    hf_errors[~hf_errors["correct"]].to_csv(
        OUTPUT_DIR / "distilbert_pair_sentiment_misclassified_examples.csv", index=False
    )


In [ ]:
# =========================
# 16. Overall model comparison
# =========================
all_results = [logreg_metrics, svc_metrics]
if HF_AVAILABLE:
    all_results.append(distilbert_metrics)

results_df = pd.DataFrame(all_results).sort_values(
    by=["macro_f1", "accuracy"], ascending=False
).reset_index(drop=True)

display(results_df)
results_df.to_csv(OUTPUT_DIR / "all_model_comparison.csv", index=False)

plt.figure(figsize=(8, 4))
plt.bar(results_df["model"], results_df["macro_f1"])
plt.ylabel("Macro F1")
plt.title("Task 2: Pair-Level Sentiment Model Comparison")
for i, v in enumerate(results_df["macro_f1"]):
    plt.text(i, v, f"{v:.3f}", ha="center", va="bottom")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "all_models_macro_f1.png", dpi=200, bbox_inches="tight")
plt.show()


## Optional: Temporal analysis using the best sentiment model's predictions

This section gives you a lightweight way to summarize sentiment over time by product.

It uses **gold test labels** by default.  
If you want, you can later adapt it to use **model predictions on the full pair-level dataset**.


In [ ]:
# =========================
# 17. Temporal summary on labeled data
# =========================
if "month" in df.columns:
    temp = df.copy()
    temp_summary = (
        temp.groupby(["month", "product", "sentiment"])
        .size()
        .reset_index(name="n")
    )

    total_by_month_product = (
        temp.groupby(["month", "product"])
        .size()
        .reset_index(name="total")
    )

    temp_summary = temp_summary.merge(total_by_month_product, on=["month", "product"], how="left")
    temp_summary["pct"] = temp_summary["n"] / temp_summary["total"]

    display(temp_summary.head(20))
    temp_summary.to_csv(OUTPUT_DIR / "temporal_sentiment_by_product.csv", index=False)

    for prod in FINAL_PRODUCTS:
        sub = temp_summary[temp_summary["product"] == prod].copy()
        if sub.empty:
            continue

        pivot = sub.pivot(index="month", columns="sentiment", values="pct").fillna(0)
        pivot = pivot.sort_index()

        plt.figure(figsize=(10, 4))
        for sent in FINAL_SENTIMENTS:
            if sent in pivot.columns:
                plt.plot(pivot.index, pivot[sent], marker="o", label=sent)
        plt.title(f"Sentiment over Time - {prod}")
        plt.xlabel("Month")
        plt.ylabel("Share")
        plt.xticks(rotation=45)
        plt.legend()
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / f"temporal_{prod}_sentiment.png", dpi=200, bbox_inches="tight")
        plt.show()
else:
    print("No date column available, so temporal summary was skipped.")


## Save prediction tables

These files are useful for your report and slide deck.


In [ ]:
# =========================
# 18. Save test predictions
# =========================
pred_tables = []

# Logistic
tmp = test_df.copy()
tmp["model"] = "tfidf_logreg"
tmp["true_label"] = [id2label[i] for i in y_test]
tmp["pred_label"] = [id2label[i] for i in test_pred_lr]
if test_prob_lr is not None:
    tmp["pred_confidence"] = np.max(test_prob_lr, axis=1)
pred_tables.append(tmp)

# SVC
tmp = test_df.copy()
tmp["model"] = "tfidf_calibrated_linearsvc"
tmp["true_label"] = [id2label[i] for i in y_test]
tmp["pred_label"] = [id2label[i] for i in test_pred_svc]
tmp["pred_confidence"] = np.max(test_prob_svc, axis=1)
pred_tables.append(tmp)

# DistilBERT
if HF_AVAILABLE:
    tmp = test_df.copy()
    tmp["model"] = "distilbert_pair_sentiment"
    tmp["true_label"] = [id2label[i] for i in y_true_hf]
    tmp["pred_label"] = [id2label[i] for i in y_pred_hf]
    tmp["pred_confidence"] = np.max(y_prob_hf, axis=1)
    pred_tables.append(tmp)

all_preds = pd.concat(pred_tables, axis=0, ignore_index=True)
all_preds.to_csv(OUTPUT_DIR / "test_set_predictions_all_models.csv", index=False)

print("Saved:", OUTPUT_DIR / "test_set_predictions_all_models.csv")
display(all_preds.head())


## Notes for the report / presentation

### Task definition
- Task 2 is **pair-level, product-conditioned multiclass sentiment classification**
- Labels are: `negative`, `neutral`, `positive`

### Why this is stronger than comment-level sentiment
Because one comment can contain different sentiments for different products.

### Important methodological point
You split by `text_id`, not by rows, to avoid leakage.

### Main results to report
- Accuracy
- Macro F1
- Weighted F1
- Per-class F1
- Confusion matrix
- AUROC
- ECE

### Recommended slide message
> We reformulated sentiment analysis as a product-conditioned multiclass classification problem. Instead of assigning one sentiment to an entire Reddit comment, we predict sentiment for each extracted `(text, product)` pair. This allows one comment to express different sentiments toward different cannabis product forms and better reflects the structure of real user discussions.
